# Set Ups

In [161]:
import os
# import uuid
import nest_asyncio
import asyncio
from fastapi import UploadFile
from llama_index.core import VectorStoreIndex, StorageContext, Settings, PromptTemplate
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.extractors import TitleExtractor
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.readers.docling import DoclingReader
from llama_index.core.schema import Document
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core.response_synthesizers import ResponseMode
from llama_index.core import get_response_synthesizer

import chromadb
from llama_index.core.retrievers import BaseRetriever
from typing import List, Dict, Any
import xml.etree.ElementTree as ET

from dotenv import load_dotenv
load_dotenv()

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.llms.ollama import Ollama
import torch
from llama_index.core.extractors import SummaryExtractor, KeywordExtractor
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from pathlib import Path
nest_asyncio.apply()

In [94]:
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
def llm_setting(name = "Gemini"):
    if name == "Gemini": 
        from llama_index.llms.gemini import Gemini
        from llama_index.embeddings.gemini import GeminiEmbedding
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        api_key = os.getenv("GEMINI_API_KEY")
        # define embedding model
        # embed_model = GeminiEmbedding(
        #     model_name="models/embedding-004", 
        #     api_key=api_key,
        #     embed_batch_size=10
        # )
        embed_model = GoogleGenAIEmbedding(
            model_name="text-embedding-004",
            api_key=api_key,
            embed_batch_size=100)
        # embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
        #                                    device=device,
        #                                    embed_batch_size=10)
        llm = Gemini(model_name="models/gemini-2.0-flash", 
                     temperature=0.1, 
                     max_tokens=100000, 
                     api_key=api_key)
    
    # if default -> use local model with HF embedding
    else:
        # api_key = os.getenv("HF_API_KEY")
        # define embedding model
        from llama_index.llms.ollama import Ollama
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
                                           device=device,
                                           embed_batch_size=10)
        
        llm = Ollama(model="deepseek-r1:7b", 
                     request_timeout=120.0)
    return llm, embed_model

## Global setting

In [95]:
# Use DoclingReader to load the data
reader = DoclingReader()

# Set up model
llm, embed_model = llm_setting()

# Set default LLM and embedding model 
Settings.llm = llm
Settings.embed_model = embed_model
# maybe need to define sentencesplitter chunk size
# Settings.text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=20)

chroma_client = chromadb.PersistentClient(path="./chroma_db_gemini")
reader = DoclingReader()
node_parser = MarkdownNodeParser()

/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_29706/3051935540.py:25: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  llm = Gemini(model_name="models/gemini-2.0-flash",


# a1. PDF to LLamaIndex in ChromaDB - PDD, Policy, Regional Policy

### 1.0 Helper Functions

In [96]:
### Common Functions

# get all file paths from folder
def get_file_paths(folder_path):
    """Get all file paths in a folder recursively, excluding .DS_Store files."""
    file_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file != '.DS_Store':
                file_paths.append(os.path.join(root, file))
    return file_paths

import re
def rename_file_with_prefix(file_path, prefix='forestPolicy_IND'):
    """
    Renames a file by stripping whitespace and special characters from its name
    and adding the prefix 'forestPolicy_IND'.
    
    Args:
        file_path (str): The full path to the file (e.g., '/path/to/My File@#$.pdf')
        prefix (str): prefix of the document
    
    Returns:
        str: The new file path after renaming, or None if the file doesn't exist or an error occurs.
    
    Example:
        Input: '/path/to/My File@#$.pdf'
        Output: '/path/to/forestPolicy_IND_MyFile.pdf'
    """
    try:
        # Check if the file exists
        if not os.path.isfile(file_path):
            print(f"Error: File '{file_path}' does not exist.")
            return None
        
        # Get the directory and file name
        directory = os.path.dirname(file_path)
        file_name = os.path.basename(file_path)
        
        # Split the file name into name and extension
        name, ext = os.path.splitext(file_name)
        
        # Clean the file name: remove special characters and whitespace
        cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name)  # Keep only alphanumeric characters
        cleaned_name = cleaned_name.strip()  # Ensure no leading/trailing whitespace
        
        # Create the new file name with prefix
        new_file_name = f"{prefix}_{cleaned_name}{ext}"
        
        # Create the new file path
        new_file_path = os.path.join(directory, new_file_name)
        
        # Rename the file
        os.rename(file_path, new_file_path)
        print(f"File renamed from '{file_name}' to '{new_file_name}'")
        
        return new_file_path
    
    except PermissionError:
        print(f"Error: Permission denied while renaming '{file_path}'.")
        return None
    except FileExistsError:
        print(f"Error: A file named '{new_file_name}' already exists in the directory.")
        return None
    except Exception as e:
        print(f"Error: An unexpected issue occurred while renaming '{file_path}': {e}")
        return None
    
# Function to check if file has already been processed
def file_already_processed(collection, file_name):
    """Check if a file has already been processed by searching collection metadata."""
    try:
        # Get all metadata from the collection
        all_metadata = collection.get(
            where={"file_name": file_name}
        )
        # If there are any results with this file_name, the file was already processed
        return len(all_metadata['metadatas']) > 0
    except Exception as e:
        print(f"Error checking if file was processed: {e}")
        return False

def add_document_to_collection(file_path_new_doc, collection, storage_context, file_type="policy_country", country = "Indonesia",
                               chroma_client=chroma_client, reader=reader, node_parser=node_parser):
    """
    Add a new document to an existing collection without overwriting the existing index.
    
    Args:
        file_path_new_doc (str): Path to the new PDD document
        storage_context: chromadb storage context
        file_type (str): pdd, policy_vcm, or policy_country,
        country (str): name of the country, if file_type is policy_country
        chroma_client: ChromaDB client
        reader: Document reader
        node_parser: Node parser for transformations
    
    """    
    # Check if required parameters are provided
    if chroma_client is None or reader is None or node_parser is None:
        raise ValueError("chroma_client, reader, and node_parser must be provided")
    
    # Get the file name for metadata
    file_name = os.path.basename(file_path_new_doc)
    
    if file_already_processed(collection, file_name):
        print(f"File {file_name} already processed, skipping...")
        return None
    
    # Load the new document
    try:
        new_documents = reader.load_data(Path(file_path_new_doc))
    except Exception as e:
        print(f"Error loading document {file_path_new_doc}: {e}")
        return None
    
    if not new_documents:
        print(f"No content extracted from {file_name}")
        return None
    
    # Add metadata to the new documents
    for doc in new_documents:
        if doc.metadata is None:
            doc.metadata = {}
        if file_type=='pdd':
            doc.metadata.update({
                "file_name": file_name,
                "registry": file_name.split("_")[0],
                "project_code": file_name.split("_")[1],
                "type": file_type
            })
        elif file_type== 'policy_vcm':
            doc.metadata.update({
                "file_name": file_name,
                "source": file_name.split("_")[0],
                "type": file_type
            })
        elif file_type=='policy_country':
            doc.metadata.update({
                "file_name": file_name,
                "country": country,
                "type": file_type
            })        
    # Create a new index with just the new documents, but using the existing storage context
    # This will add the new documents to the existing collection
    try:
        index = VectorStoreIndex.from_documents(
            documents=new_documents,
            transformations=[node_parser],
            storage_context=storage_context,
            cache=IngestionCache()
        )
        print(f"Successfully added {file_name} to the {collection_name} collection")
        return index
    except Exception as e:
        print(f"Error creating index for {file_name}: {e}")
        return None


#### Check up OR Delete ChromaCollection

In [82]:
collections = chroma_client.list_collections() 

# # Delete a specific collection
# chroma_client.delete_collection('chroma_collection_pdd')

# # Delete each collection
# for collection in collections:
#     chroma_client.delete_collection(collection)
#     print(f"Deleted collection: {collection}")
collections

[]

In [ ]:
# # test if the settings with embedding etc works 
# collection_pdd = chroma_client.get_or_create_collection(name="testing")
# storage_context_pdd  = StorageContext.from_defaults(
#     vector_store=ChromaVectorStore(collection_pdd) 
# )
# new_documents = reader.load_data('/Users/beckyxu/Desktop/illapaper/0-OpenAI Cofounder_ The 27 Papers to Read to Know 90% About AI.pdf')
# index_test = VectorStoreIndex.from_documents(
#     documents=new_documents,
#     transformations=[node_parser],
#     storage_context=storage_context_pdd,
#     cache=IngestionCache()
# )

# testengine = index_test.as_query_engine()
# testengine.query('what is this about').response

## 1.1 PDD Process

In [88]:
### PDD
# NOTE: Change things here
file_path_pdd = 'pdd/VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf'

In [89]:
# NOTE Change this to account for adding new pdd
file_name = os.path.basename(file_path)
# construct vector store and customize storage context
collection_pdd = chroma_client.get_or_create_collection(name="pdd_new")
storage_context_pdd  = StorageContext.from_defaults(
    vector_store=ChromaVectorStore(collection_pdd) 
)

index_pdd = add_document_to_collection(file_path_pdd, collection_pdd, storage_context_pdd, 
                                       file_type="pdd")

# OLD
# documents = reader.load_data(Path(file_path_pdd))
# for doc in documents:
#     if doc.metadata is None:
#         doc.metadata = {}
#     doc.metadata.update({
#         "file_name": file_name,
#         "registry": file_name.split("_")[0],
#         "project_code": file_name.split("_")[1],
#         "type": "pdd"
#     })
    
# index_pdd = VectorStoreIndex.from_documents(
#     documents=documents,
#     transformations=[node_parser],
#     storage_context=storage_context_pdd,
#     cache=IngestionCache()
# )

Successfully added VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf to the regional_policy_IND collection


## 1.2 Policy VCM & Regional - DONE - DON'T TOUCH - 

### VCM Policy

In [90]:
# VCM Policy documents
folder_name = "policy_vcm_docs"

file_folder_path = os.path.join(os.getcwd(), folder_name) 
collection_name = 'policy_vcm'

# Create storage context with the existing collection
collection = chroma_client.get_or_create_collection(
                            name=collection_name)
storage_context = StorageContext.from_defaults(
                            vector_store=ChromaVectorStore(collection))

for file_path in get_file_paths(file_folder_path):
    # if need to rename files
    # rename_file_with_prefix(file_path, prefix='forestPolicy_IND')
    print(f'processing {file_path}')
    file_name = os.path.basename(file_path)
    add_document_to_collection(file_path, collection, storage_context, file_type='policy_vcm')
    print(f'successfully converted {os.path.basename(file_path)} to vector')

processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_vcm_docs/CAR_Forest_V5.0_Summary_for_Landowners.pdf
Successfully added CAR_Forest_V5.0_Summary_for_Landowners.pdf to the policy_vcm collection
successfully converted CAR_Forest_V5.0_Summary_for_Landowners.pdf to vector
processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_vcm_docs/ICVCM_CCPs.pdf
Successfully added ICVCM_CCPs.pdf to the policy_vcm collection
successfully converted ICVCM_CCPs.pdf to vector
processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_vcm_docs/VERRA_VT0009-Combined-Baseline-and-Additionality-Assessment-v1.0.pdf
Successfully added VERRA_VT0009-Combined-Baseline-and-Additionality-Assessment-v1.0.pdf to the policy_vcm collection
successfully converted VERRA_VT0009-Combined-Baseline-and-Additionality-Assessment

In [ ]:
collection_name = "policy_vcm"
# Create storage context with the existing collection
collection = chroma_client.get_or_create_collection(
                            name=collection_name)

# VERRA_Tool_demonstration_and_assessment_of_additionality .pdf

### National & Regional Policy

In [91]:
# National & Regional Policy and Report document
# NOTE Change the project directory here
folder_name = "policy_indo_docs"

file_folder_path = os.path.join(os.getcwd(), folder_name) 
collection_name = "regional_policy_IND"

# Create storage context with the existing collection
collection = chroma_client.get_or_create_collection(
                            name=collection_name)
storage_context = StorageContext.from_defaults(
                            vector_store=ChromaVectorStore(collection))

for file_path in get_file_paths(file_folder_path):
    # if need to rename files
    # rename_file_with_prefix(file_path, prefix='forestPolicy_IND')
    print(f'processing {file_path}')
    add_document_to_collection(file_path, collection, storage_context, file_type='policy_country')
    print(f'successfully converted {os.path.basename(file_path)} to vector')

processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_indo_docs/forestPolicy_IND_FirstNDCIndonesiasubmittedtoUNFCCCSetNovember2016.pdf
Successfully added forestPolicy_IND_FirstNDCIndonesiasubmittedtoUNFCCCSetNovember2016.pdf to the regional_policy_IND collection
successfully converted forestPolicy_IND_FirstNDCIndonesiasubmittedtoUNFCCCSetNovember2016.pdf to vector
processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_indo_docs/forestPolicy_IND_PPNomor23Tahun2021.pdf
Successfully added forestPolicy_IND_PPNomor23Tahun2021.pdf to the regional_policy_IND collection
successfully converted forestPolicy_IND_PPNomor23Tahun2021.pdf to vector
processing /Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline/policy_indo_docs/forestPolicy_IND_CIFORICRAFWP14.pdf
Successfully added forestPolicy_IND_CIFORICRAFWP14.p

### Sanity Check

In [ ]:
# # Check number of results
# chroma_collection_pdd = chroma_client.get_or_create_collection(name="pdd")
# result = chroma_collection_pdd.get()

# nodes = []
# if result and 'ids' in result and len(result['ids']) > 0:
#     for i, node_id in enumerate(result['ids']):
#         node_info = {
#             'id': node_id,
#             'text': result['documents'][i] if 'documents' in result else None,
#             'metadata': result['metadatas'][i] if 'metadatas' in result else None
#         }
#         nodes.append(node_info)
        
# print(f"Retrieved {len(nodes)} nodes from collection")

Retrieved 115 nodes from collection


# a2. LLM Service

#### 2.0 Global Setting & helper functions

In [ ]:
# helper function:
import requests
import xml.etree.ElementTree as ET
from typing import Dict, Any
import google.generativeai as genai

API_URL = os.getenv("LLM_API_URL", "http://localhost:11434/api/generate") 
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

def call_llm_api(prompt: str) -> str:
    """
    Call Gemini LLM API with the provided prompt
    Args:
        prompt (str): The prompt to send to the LLM
    Returns:
        str: The LLM's response text
    """
    try:
        # Initialize the Gemini model (adjust model name as needed)
        model = genai.GenerativeModel(
            model_name="gemini-2.0-flash",  
            generation_config={
                "temperature": 0.0,        # Controls randomness (0.0 to 1.0)
                "max_output_tokens": 50000, # Max tokens in response
            }
        )

        # Generate content with the prompt
        response = model.generate_content(prompt)

        # Check if response was blocked or empty
        if not response.text:
            raise Exception("Gemini API returned no valid response")

        return response.text

    except Exception as e:
        raise Exception(f"Gemini API error: {str(e)}")

def parse_xml_response(response: str, root_tag: str) -> Dict[str, Any]:
    """
    Parse XML response from LLM into a dictionary, handling potential extra text.
    
    Args:
        response (str): Raw LLM response containing XML
        root_tag (str): Expected root tag (e.g., "project_info")
    
    Returns:
        Dict[str, Any]: Parsed XML as a dictionary
    """
    # Try to find any XML-like structure if the exact root_tag isn't found
    xml_start = response.find(f"<{root_tag}>")
    xml_end = response.rfind(f"</{root_tag}>")
    
    if xml_start == -1 or xml_end == -1:
        # Fallback: Look for any XML root tag (e.g., <information>)
        possible_start = response.find("<")
        possible_end = response.rfind(">")
        if possible_start != -1 and possible_end != -1 and possible_end > possible_start:
            xml_content = response[possible_start:possible_end + 1]
        else:
            raise Exception("Could not find XML in LLM response")
    else:
        xml_content = response[xml_start:xml_end + len(f"</{root_tag}>")]

    # Parse XML
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError as e:
        raise Exception(f"Invalid XML in LLM response: {e}")

    # Convert to dictionary recursively
    def xml_to_dict(element):
        result = {}
        # Handle attributes
        if element.attrib:
            result["@attributes"] = element.attrib
        # Handle children
        for child in element:
            child_data = xml_to_dict(child)
            if child.tag in result:
                if not isinstance(result[child.tag], list):
                    result[child.tag] = [result[child.tag]]
                result[child.tag].append(child_data)
            else:
                result[child.tag] = child_data
        # Handle text content
        text = element.text.strip() if element.text else ""
        if text and not result:
            return text
        elif text:
            result["#text"] = text
        return result

    parsed_dict = xml_to_dict(root)
    
    # If root tag doesn't match expected, warn but proceed
    if root.tag != root_tag:
        print(f"Warning: Expected root tag '{root_tag}', found '{root.tag}'. Proceeding with parsed data.")
    
    return parsed_dict


## 2.1 Function A: PDD Project Basic Info Extraction

In [ ]:
def extract_doc_basicInfo(collection_name: str, project_code: str) -> Dict[str, Any]:
    """
    Extract basic project information from document using LLM with XML-formatted output
    Args:
        collection_name (str): chroma collection name of pdd 
        project_code (str): project code 
    Returns:
        Dictionary containing extracted project information
    """
    
    pdd_collection = chroma_client.get_or_create_collection(collection_name)
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    index_pdd = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)

    retriever = index_pdd.as_retriever(similarity_top_k=20, metadata_filters={"project_code": project_code})  # Retrieve top 10 most relevant chunks
    
    query = "Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project;Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)"
    retrieved_nodes = retriever.retrieve(query)
    
    # Format retrieved documents into context
    retrieved_texts = "\n\n".join([node.text for node in retrieved_nodes])
    # print(retrieved_texts)
    prompt = f"""
    Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project; Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)

    ### Instructions ###
    - Extract these exact fields from the document: project name, description, location, coordinates, status, start date, end date, methodology, size.
    - Use the *exact* XML tag names as shown in the Output Format: <name>, <description>, <location>, <status>, <start_date>, <end_date>, <methodology>, <size>.
    - Output *ONLY* the XML structure—do not include any additional text, comments, `<think>` tags, markdown (```xml```), or explanations before or after the XML.
    - Wrap the output in the root tag `<project_info>`.
    - Keep <project_code> as it is 
    - If a field is missing or not found, use "Not specified" as the value. Infer project name if project name is not found.
    - Ensure the XML is well-formed and matches the Output Format exactly in structure and tag names.

    ### Output Format ###
    <project_info>
      <project_code>{project_code}<project_code>
      <name>PROJECT TITLE</name>
      <description>BRIEF DESCRIPTION</description>
      <location>LOCATION</location>
      <status>STATUS</status>
      <start_date>START DATE</start_date>
      <end_date>END DATE</end_date>
      <methodology>METHODOLOGY</methodology>
      <size>SIZE</size>
    </project_info>

    ### Document to Analyze ###
    {retrieved_texts}

    ### Final Directive ###
    Return ONLY the XML below, using the exact tag names from the Output Format, with no deviations or additional content.
    """
    
    # Call your preferred LLM API
    response = call_llm_api(prompt)
    # print(response)
    # Parse XML response
    try:
        parsed_data = parse_xml_response(response, "project_info")
        return parsed_data
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        raise Exception(f"Failed to parse LLM output: {e}")


#### Result

In [ ]:
# Result
project_data = extract_doc_basicInfo(project_code='3226')
project_data

```xml
<project_info>
  <project_code>3226</project_code>
  <name>Padang Tikar Landscape REDD+ Project</name>
  <description>REDD+ project focused on reducing GHG emissions from unplanned deforestation and wetland degradation through conservation and sustainable management activities.</description>
  <location>Padang Tikar Landscape, Kubu Raya, West Kalimantan, Indonesia</location>
  <status>Under development</status>
  <start_date>August 30, 2017</start_date>
  <end_date>August 29, 2047</end_date>
  <methodology>VM0007 REDD+ Methodology Framework (REDD+MF), Version 1.6</methodology>
  <size>58,672.7 ha</size>
</project_info>
```


{'project_code': '3226',
 'name': 'Padang Tikar Landscape REDD+ Project',
 'description': 'REDD+ project focused on reducing GHG emissions from unplanned deforestation and wetland degradation through conservation and sustainable management activities.',
 'location': 'Padang Tikar Landscape, Kubu Raya, West Kalimantan, Indonesia',
 'status': 'Under development',
 'start_date': 'August 30, 2017',
 'end_date': 'August 29, 2047',
 'methodology': 'VM0007 REDD+ Methodology Framework (REDD+MF), Version 1.6',
 'size': '58,672.7 ha'}

## 2.2 Function B: PDD vs VCM_Policy Risk Analysis

In [ ]:
# Change model to gemini
llm_g, _ = llm_setting("Gemini")
# Set default LLM and embedding model 
# Settings.llm = llm

/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_1286/3485569342.py:16: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  llm = Gemini(model_name="models/gemini-1.5-flash", temperature=0.1, max_tokens=50000, api_key=api_key)


In [ ]:
def analyze_project_risks(project_code: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Analyze project risks by comparing PDD against policy documents, returning XML-structured results.

    Args:
        project_code: project code 
        top_k: Number of top documents to retrieve.

    Returns:
        List of dictionaries containing risk metrics per query.
    """
    # Setup Chroma connections
    chroma_client = chromadb.PersistentClient(path="./chroma_db_new")
    pdd_collection = chroma_client.get_or_create_collection("pdd")
    policy_collection = chroma_client.get_or_create_collection("vcm_policy")

    # Create vector stores and indexes
    from llama_index.vector_stores.chroma import ChromaVectorStore
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
    pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)

    # Custom dual retriever
    class DualRetriever(BaseRetriever):
        def __init__(self, pdd_retriever, policy_retriever):
            self.pdd_retriever = pdd_retriever
            self.policy_retriever = policy_retriever
            super().__init__()

        def _retrieve(self, query, **kwargs):
            pdd_nodes = self.pdd_retriever.retrieve(query)
            policy_nodes = self.policy_retriever.retrieve(query)
            # Add metadata to nodes
            for node in pdd_nodes:
                node.node.metadata["source"] = "project_design_document"
            for node in policy_nodes:
                node.node.metadata["source"] = "industry_standard"
            # Return a single list combining both sets of nodes
            return pdd_nodes + policy_nodes

    filters = MetadataFilters(filters=[
    ExactMatchFilter(
        key="project_code", 
        value=project_code
        )
    ])
    pdd_retriever = pdd_index.as_retriever(similarity_top_k=top_k, filters= filters)
    policy_retriever = policy_index.as_retriever(similarity_top_k=top_k)
    dual_retriever = DualRetriever(pdd_retriever, policy_retriever)

    # Custom prompt for XML output
    risk_template_str = (
        "Analyze the risk profile of a carbon offset project by comparing its project design document (PDD) "
        "with established carbon offset policy documents for the aspect: {query}.\n\n"
        "<instructions>\n"
        "- Review the PDD content: {pdd_texts}\n"
        "- Compare it against policy standards: {policy_texts}\n"
        "- Identify potential risks in the category listed in < > in query.\n"
        "- Assign an overall risk score (0-100), impact level (Low, Medium, High), and likelihood (Unlikely, Possible, Likely).\n"
        "- Provide a brief description for the risks.\n"
        "- Provide a list of keywords, separated by comma, that describe the risks\n"
        "- Use the *exact* XML tag names as listed in after query's analyze the risk category:.\n"
        "- Output *ONLY* one <risk_category>\. If there are multiple risks, then explain in the description.\n"
        "- Output *ONLY* the XML structure—do not include additional text, comments, `<think>` tags, markdown, or explanations.\n"
        "</instructions>\n\n"
        "<output_format>\n"
        "<risk_metrics>\n"
        "  <risk_category name=\"CATEGORY\">\n"
        "    <score>SCORE_VALUE</score>\n"
        "    <impact>IMPACT_LEVEL</impact>\n"
        "    <likelihood>LIKELIHOOD</likelihood>\n"
        "    <description>RISK_DESCRIPTION</description>\n"
        "    <keywords>RISK_KEYWORDS</keywords>\n"
        "  </risk_category>\n"
        "</risk_metrics>\n"
        "</output_format>"
    )
    risk_template = PromptTemplate(risk_template_str)

    all_risk_metrics = []
    
    project_query_list = ["analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes.",
                      "analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type.",
                      "analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about safeguards like buffer pools, long-term management plans, or insurance against reversals (e.g., due to fires or deforestation).",
                      "analyze the risk category: <Leakage> - Has the project assessed potential leakage, and how is it accounted for in the emissions reductions calculations? Leakage occurs when emissions are displaced elsewhere (e.g., deforestation shifting to another area). Verify if a leakage assessment was conducted and if mitigation measures are included.",
                      "analyze the risk category: <Monitoring and Verification> - What is the monitoring plan, and how will the project's emissions reductions be verified by a third party? A robust monitoring plan should detail what data will be collected, how, and how often. Confirmation of third-party verification ensures accuracy and independence."
                     ]
    
    # Process each query using risk_query_engine
    for query in project_query_list:
        print(f"Analyzing: {query}")
        # Retrieve nodes using the dual retriever
        nodes = dual_retriever.retrieve(query)
        # Separate nodes by source for context
        pdd_nodes = [node for node in nodes if node.node.metadata.get("source") == "project_design_document"]
        policy_nodes = [node for node in nodes if node.node.metadata.get("source") == "industry_standard"]
        # Combine retrieved content into context
        pdd_context_str = "\n\n".join([node.node.get_content() for node in pdd_nodes])
        policy_context_str = "\n\n".join([node.node.get_content() for node in policy_nodes])
        # Format the full prompt with context
        formatted_prompt = risk_template.format(
            query=query,
            project_code=project_code,
            pdd_texts=pdd_context_str,
            policy_texts=policy_context_str
        )
        # Call the LLM directly
        response_text = call_llm_api(formatted_prompt)  # Assuming this function is defined elsewhere
        try:
            # Parse XML using the new function
            parsed_response = parse_xml_response(response_text, "risk_metrics")
            risk_metrics = []
            
            # Handle case where risk_category is a single dict or a list
            risk_categories = parsed_response.get("risk_category")
            if not risk_categories:
                print(f"No risk categories found for '{query}'")
                continue
            if isinstance(risk_categories, dict):
                risk_categories = [risk_categories]
                
            for category in risk_categories:
                risk_metrics.append({
                    "category": category["@attributes"]["name"],
                    "score": int(category["score"]),
                    "impact": category["impact"],
                    "likelihood": category["likelihood"],
                    "description": category["description"],
                    # "query": query
                })
            all_risk_metrics.extend(risk_metrics)
            # print(f"Risk metrics for '{query}': {response_text}")
        except Exception as e:
            print(f"Error parsing risk metrics for '{query}': {e}")
            raise Exception(f"Failed to parse risk metrics: {e}")

    return all_risk_metrics

<>:65: SyntaxWarning: invalid escape sequence '\.'
<>:65: SyntaxWarning: invalid escape sequence '\.'
/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_1286/370823593.py:65: SyntaxWarning: invalid escape sequence '\.'
  "- Output *ONLY* one <risk_category>\. If there are multiple risks, then explain in the description.\n"


#### Result

In [ ]:
project_code = '3226'
all_risk_metrics = analyze_project_risks(project_code)
all_risk_metrics

Analyzing: analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes.
Analyzing: analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type.
Analyzing: analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about saf

[{'category': 'Additionality',
  'score': 65,
  'impact': 'High',
  'likelihood': 'Possible',
  'description': "The project's additionality relies on demonstrating that conservation would not occur without carbon credit financing due to historical non-compliance with conservation measures, insufficient economic incentives for local communities, and poverty levels. The risk lies in the potential overestimation of the baseline deforestation rate or underestimation of alternative conservation efforts that could occur independently. While the PDD mentions the use of tools for demonstrating additionality, the strength of the evidence and the rigor of applying these tools are critical. Policy documents emphasize the need to demonstrate that reductions are above and beyond legal requirements and business-as-usual activities. A key risk is that the project area might have been subject to conservation efforts regardless, or that the economic barriers are not as significant as claimed. The relia

## 2.3 Function C: PDD vs Regional Policy Risk Analysis

In [ ]:
# TODO:
# Change the risk template to output these items:
# risk_policy = {
#     "summary": {
#         "overall_summary": "This is a summary of the project risks and compliance with policies.",
#         "recommendations": [
#             {"action": "Strengthen additionality evidence"},
#             {"action": "Improve leakage monitoring"},
#             {"action": "Enhance permanence safeguards"}
#         ],
#         "additional_insights": "Additional insights about the project..."
#     }
# }

def analyze_regional_risks(project_code: str, collection_name_policy =regional_policy_IND, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Analyze project risks by comparing PDD against policy documents, returning XML-structured results.

    Args:
        project_code: project code 
        collection_name (str): name of chroma collection containing regional policy
        top_k: Number of top documents to retrieve.

    Returns:
        List of dictionaries containing risk metrics per query.
    """
    # Setup Chroma connections
    pdd_collection = chroma_client.get_or_create_collection("pdd_new")
    policy_collection = chroma_client.get_or_create_collection(collection_name_policy)

    # Create vector stores and indexes
    from llama_index.vector_stores.chroma import ChromaVectorStore
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
    pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)

    # Custom dual retriever
    class DualRetriever(BaseRetriever):
        def __init__(self, pdd_retriever, policy_retriever):
            self.pdd_retriever = pdd_retriever
            self.policy_retriever = policy_retriever
            super().__init__()

        def _retrieve(self, query, **kwargs):
            pdd_nodes = self.pdd_retriever.retrieve(query)
            policy_nodes = self.policy_retriever.retrieve(query)
            # Add metadata to nodes
            for node in pdd_nodes:
                node.node.metadata["source"] = "project_design_document"
            for node in policy_nodes:
                node.node.metadata["source"] = "national_policy_document"
            # Return a single list combining both sets of nodes
            return pdd_nodes + policy_nodes

    # TODO: change the metadata filter operation
    
    # from llama_index.vector_stores.types import ExactMatchFilter, MetadataFilters

    # # Insert a document with specific metadata
    # doc = Document(text="target", metadata={"tag": "target"})
    # index.insert(doc)

    # # Create a filter that matches the inserted metadata
    # filters = MetadataFilters(
    #     filters=[ExactMatchFilter(key="tag", value="target")]
    # )

    # # Use the filter in the retriever to retrieve only the documents that match the filter
    # retriever = index.as_retriever(
    #     similarity_top_k=20,
    #     filters=filters,
    # )
    
    pdd_retriever = pdd_index.as_retriever(similarity_top_k=top_k, metadata_filters={"project_code": project_code})
    policy_retriever = policy_index.as_retriever(similarity_top_k=top_k)
    dual_retriever = DualRetriever(pdd_retriever, policy_retriever)

    # Custom prompt for XML output
    risk_template_str = (
        "Analyze the risk profile of a carbon offset project by comparing its project design document (PDD)"
        "with established national and regional forestry policy documents for the aspect: {query}.\n\n"
        "here is the context of the {pdd_risk_texts}"
        "<instructions>\n"
        "- Review the PDD content: {pdd_texts}\n"
        "- Compare it against policy standards: {policy_texts}\n"
        "- Identify potential risks in the category listed in < > in query.\n"
        "- Output *ONLY* the XML structure—do not include additional text, comments, `<think>` tags, markdown, or explanations.\n"
        "</instructions>\n\n"
        "<output_format>\n"
        "<risk_metrics>\n"
        "  <risk_category name=\"CATEGORY\">\n"
        "    <description>description of the risk</description>\n"
        "    <summary>summary of the risk</summary>\n"
        "    <recommendations>recommendations for the risk"
        "        <action>action to take to further evaluate the risk</action>"
        "    </recommendations>\n"
        "    <keywords>RISK_KEYWORDS</keywords>\n"
        "  </risk_category>\n"
        "</risk_metrics>\n"
        "</output_format>"
    )
    risk_template = PromptTemplate(risk_template_str)

    all_risk_metrics = []
    
    project_query_list = ["analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes.",
                      "analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type.",
                      "analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about safeguards like buffer pools, long-term management plans, or insurance against reversals (e.g., due to fires or deforestation).",
                      "analyze the risk category: <Leakage> - Has the project assessed potential leakage, and how is it accounted for in the emissions reductions calculations? Leakage occurs when emissions are displaced elsewhere (e.g., deforestation shifting to another area). Verify if a leakage assessment was conducted and if mitigation measures are included.",
                      "analyze the risk category: <Monitoring and Verification> - What is the monitoring plan, and how will the project's emissions reductions be verified by a third party? A robust monitoring plan should detail what data will be collected, how, and how often. Confirmation of third-party verification ensures accuracy and independence."
                     ]
    
    # Process each query using risk_query_engine
    for query in project_query_list:
        print(f"Analyzing: {query}")
        # Create a risk summary context from PDD
        pdd_risk_nodes = pdd_index.as_query
        # Retrieve nodes using the dual retriever
        nodes = dual_retriever.retrieve(query)
        # Separate nodes by source for context
        pdd_nodes = [node for node in nodes if node.node.metadata.get("source") == "project_design_document"]
        policy_nodes = [node for node in nodes if node.node.metadata.get("source") == "industry_standard"]
        # Combine retrieved content into context
        pdd_context_str = "\n\n".join([node.node.get_content() for node in pdd_nodes])
        policy_context_str = "\n\n".join([node.node.get_content() for node in policy_nodes])
        # Format the full prompt with context
        formatted_prompt = risk_template.format(
            query=query,
            pdd_risk_texts = pdd_risk_texts,
            project_code=project_code,
            pdd_texts=pdd_context_str,
            policy_texts=policy_context_str
        )
        # Call the LLM directly
        response_text = call_llm_api(formatted_prompt)  # Assuming this function is defined elsewhere
        try:
            # Parse XML using the new function
            parsed_response = parse_xml_response(response_text, "risk_metrics")
            risk_metrics = []
            
            # Handle case where risk_category is a single dict or a list
            risk_categories = parsed_response.get("risk_category")
            if not risk_categories:
                print(f"No risk categories found for '{query}'")
                continue
            if isinstance(risk_categories, dict):
                risk_categories = [risk_categories]
                
            for category in risk_categories:
                risk_metrics.append({
                    "category": category["@attributes"]["name"],
                    "score": int(category["score"]),
                    "impact": category["impact"],
                    "likelihood": category["likelihood"],
                    "description": category["description"],
                    # "query": query
                })
            all_risk_metrics.extend(risk_metrics)
            # print(f"Risk metrics for '{query}': {response_text}")
        except Exception as e:
            print(f"Error parsing risk metrics for '{query}': {e}")
            raise Exception(f"Failed to parse risk metrics: {e}")

    return all_risk_metrics

#### Result - simulated 

In [ ]:
# project_code = '3226'
# regional_risk_metrics = analyze_regional_risks(project_code)
# regional_risk_metrics

# # Result - Simulated:
# risk_policy = {
#     "summary": {
#         "overall_summary": "This is a summary of the project risks and compliance with policies.",
#         "recommendations": [
#             {"action": "Strengthen additionality evidence"},
#             {"action": "Improve leakage monitoring"},
#             {"action": "Enhance permanence safeguards"}
#         ],
#         "additional_insights": "Additional insights about the project..."
#     }
# }

# Function to upload!

In [ ]:
from database import store_analysis_results
async def upload_to_supabase():
    project_id = await store_analysis_results(project_data, all_risk_metrics, risk_policy)
    print(f"Successfully uploaded project data with ID: {project_id}")
    # Run the async function
asyncio.run(upload_to_supabase())

Successfully uploaded project data with ID: 084650fe-b1f5-44d4-9a40-a7d4e7f42f30


### New Pipeline for policy analysis
1. extract project info from PDD -> {pdd_info}
    pdd_info = 
    {
        “project_overiew”: “extracted information”, 
        "current_landuse": "extracted information"
        "location": "extracted information",
        "claimed_baseline_threats": "extracted information",
        "claimed_revenue_sources": "extracted information",
        "claimed_additionality": "extracted information",
    }
2. extract policy info from policy_vcm -> {policy_info}
    context: {pdd_info}
    expected output: 
    policy_info =
    {
        General: "question"
        Financial: "questions"
        Regulatory: "questions"
        Implementaiton: "questions"
    }

3. agent gap filler and social research -- NEXT STEP


In [170]:
def extract_clean_json(response_str):
    # Remove markdown formatting
    json_text = re.sub(r"^```json|```$", "", response_str.strip(), flags=re.MULTILINE)
    return json.loads(json_text)

def clean_text(text: str) -> str:
    # Remove font glyph artifacts
    cleaned = re.sub(r"glyph<c=\d+,font=[^>]+>", "", text)
    # Remove non-printable characters
    cleaned = re.sub(r"[^\x20-\x7E\n\r]", "", cleaned)
    return cleaned

In [ ]:
from typing import Dict, List, Any, Optional
import re

def analyze_regional_risks(project_code: str, 
                           collection_name_policy="regional_policy_IND", 
                           top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Analyze project risks by comparing PDD against policy documents, returning XML-structured results.

    Args:
        project_code: project code 
        collection_name (str): name of chroma collection containing regional policy
        top_k: Number of top documents to retrieve.

    Returns:
        List of dictionaries containing risk metrics per query.
    """
    # --------- Setup Chroma connections
    pdd_collection = chroma_client.get_or_create_collection("pdd_new")

    # Create vector stores and indexes
    from llama_index.vector_stores.chroma import ChromaVectorStore
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    # policy_collection = chroma_client.get_or_create_collection(collection_name_policy)
    # policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
    # policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)
    
    # --------- step 1: query pdd project basic info
    filters = MetadataFilters(filters=[
    ExactMatchFilter(
        key="project_code", 
        value=project_code
        )
    ])
    
    pdd_retriever = pdd_index.as_retriever(filters=filters, similarity_top_k=50)
    response_synthesizer = get_response_synthesizer()

    # assemble query engine
    pdd_query_engine = RetrieverQueryEngine(
        retriever=pdd_retriever,
        response_synthesizer=response_synthesizer,
        # node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.7)],
    )
    pdd_extraction_prompt = """
        You are an AI assistant tasked with extracting specific information from a Project Design Document (PDD). Your goal is to output a JSON object with the following fixed keys, ensuring that each key is present in the output. If a particular piece of information is not available in the PDD, use "Not specified" as the value.

        Please adhere strictly to the following JSON structure:

        {
        "project_overview": "Provide a concise summary of the project's objectives and key design elements as a carbon offset initiative.",
        "current_landuse": "Describe the designated land use of the project area prior to implementation.",
        "location": "Specify the project's location, including the province or region. return a list [name of location, province]",
        "local_economy": "Summarize the characteristics of the local economy in the project area.",
        "baseline_scenario": "Detail the baseline scenario, outlining what would occur in the absence of the project, and how this baseline is created using what methodologies",
        "justification_of_additionality": "How does the project demonstrate that the carbon benefits would not occur without its implementation?",
        "permanence": "How does the project ensure long-term carbon sequestration, and what measures are in place to address potential reversals (e.g., wildfires, logging)?"
        "community_engagement": "How were local communities consulted during project development, and do they have mechanisms for ongoing participation?  Explain."
        "benefit_sharing": "Does the project provide tangible benefits to local stakeholders, such as employment or revenue sharing?  Explain."
        "rights_and_land_tenure": "Are land rights and tenure issues clearly addressed, ensuring that the project does not infringe upon indigenous or local communities' rights? Explain."
        "revenue_streams": "What are the projected revenue sources, and are they diversified to ensure financial stability?",        
        }

        Ensure that:
        - All keys are included in the output JSON.
        - The values are extracted directly from the PDD content.
        - The output is a valid JSON object without any explanatory text or commentary.
        """
        
    ## if you wanna find the retrieve nodes     
    # retrieved_nodes = pdd_retriever.retrieve(pdd_extraction_prompt)
    # Format retrieved documents into context
    # retrieved_texts = "\n\n".join([node.text for node in retrieved_nodes])
    # print(f'node are: {retrieved_texts}')

    pdd_basicinfo = pdd_query_engine.query(pdd_extraction_prompt)
    
    # Example usage
    clean_data = extract_clean_json(pdd_basicinfo.response)  
    
    cleaned_json = {k: clean_text(v) for k, v in clean_data.items()}

    # clean_json is the input to the next stemp
    # return cleaned_json


In [157]:
from llama_index.core.query_engine import RetrieverQueryEngine
response = analyze_regional_risks("3226")
response

{'project_overview': 'The Padang Tikar Landscape project aims to reduce GHG emissions from deforestation by balancing protection, production, and inclusion in Village Forests. It seeks to secure and protect the Village Forest of Padang Tikar and restore and improve ecosystem services and habitat through conservation activities, community development, and improved livelihoods.',
 'current_landuse': 'The PA corresponds to secondary dryland forest, secondary swamp forest, and secondary mangrove forest within the Padang Tikar Landscape that have remained as forests throughout the period 2006  2016.',
 'location': "['Padang Tikar Landscape', 'West Kalimantan']",
 'local_economy': 'Before the project began, illegal logging in mangroves and agricultural areas was the most widespread land use in the project area (PA), contributing to growing insecurity, a lack of education, and limited employment opportunities. 38% of local communities depend on fishery sectors for their livelihoods.',
 'basel

In [174]:
# STEP 2: 
collection_name_policy = "regional_policy_IND" # remove
policy_collection = chroma_client.get_or_create_collection(collection_name_policy)
policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)
policy_retriever = policy_index.as_retriever(similarity_top_k=10)
response_synthesizer = get_response_synthesizer()
# assemble query engine
policy_query_engine = RetrieverQueryEngine(
    retriever=policy_retriever,
    response_synthesizer=response_synthesizer,
    # node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.7)],
    )

# Custom prompt for XML output
risk_template_str = (
    """Analyze the risk profile of a carbon offset project by comparing its project design document (PDD)
    with established national and regional forestry policy documents for the aspect: {query}.
    Here are some context of the {pdd_risk_texts} from the PDD. If there are abbreviations, list the full names in a bracket.
    
    Use the *exact* tag name in the XML tag listed in after query's analyze the risk category.
    For example, the query starts with "risk cateogry: <general> - question", the expected output is {"general":"query outputs"}
    
    Ensure that:
    - All keys are included in the output JSON.
    - The output is a valid JSON object
    """
)
risk_template = PromptTemplate(risk_template_str)

all_risk_metrics = []

project_query_list = [
    "risk category: <regulatory> - Are forest conservation, mangrove restoration, or ecosystem services programs already supported in [location/province] through national or regional plans? Does Indonesian forestry policy already include similar forest protection efforts in this area? What are the official land use classifications and restrictions for [location/province]? Is the project area already protected under Indonesian forestry or conservation law / Is [project location] part of a moratorium, protected forest, or conservation area under national/regional law? Are there reforestation, REDD+, or forest protection programs already underway here?",
    "risk category: <finance> - Do government grants, public incentives, or subsidies already exist for forest protection or restoration in this area? Does policy or funding from MoEF or local government already enable these project activities regardless of carbon financing?",
    "risk category: <permanence> - Does the government have forest patrols, peatland restoration, or wildfire prevention programs in this area? What regulations are in place to safeguard protected forests in [location]?",
    "risk category: <local_economy> - Are there national or local programs addressing economic issues(e.g. illegal logging, unsustainable agriculture) in [location]? Are there alternative livelihood programs supported by the government in this region?",
    ]

# Process each query using risk_query_engine
for query in project_query_list:
    # Format the full prompt with context
    formatted_prompt = risk_template.format(
        query=query,
        pdd_risk_texts=response,
    )
    # Call the LLM directly
    response_text = policy_query_engine.query(formatted_prompt)  # Assuming this function is defined elsewhere
    print(response_text)
    all_risk_metrics.append(response_text.response)
    
def merge_json_outputs(json_strings):
    """
    Merge multiple JSON strings into a single JSON object.
    
    Args:
        json_strings (list): List of JSON strings to merge
        
    Returns:
        dict: Merged JSON object
    """
    result = {}
    
    for json_str in json_strings:
        # Remove markdown code block formatting if present
        clean_str = json_str.replace("```json", "").replace("```", "").strip()
        
        try:
            # Parse the JSON string
            data = json.loads(clean_str)
            
            # Merge with the result
            result.update(data)
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON: {e}")
            print(f"Problematic JSON string: {clean_str}")
    
    return result

# Merge the JSON strings
merged_json = merge_json_outputs(all_risk_metrics)

# Print the merged JSON
print(json.dumps(merged_json, indent=2))

# Save to file
with open("merged_regional_risks.json", "w") as f:
    json.dump(merged_json, f, indent=2)

print("Merged JSON saved to merged_regional_risks.json")


```json
{
  "regulatory": "The Padang Tikar Landscape project in West Kalimantan aims to reduce GHG emissions through forest conservation, mangrove restoration, and ecosystem services, aligning with potential national and regional plans supporting similar initiatives. To assess regulatory risks, it's important to determine if forest conservation, mangrove restoration, or ecosystem services programs are already supported in West Kalimantan through national or regional plans. Also, it should be verified whether Indonesian forestry policy already includes similar forest protection efforts in this area. Official land use classifications and restrictions for West Kalimantan need to be checked, and it should be confirmed whether the project area is already protected under Indonesian forestry or conservation law. It is also important to determine if the project location is part of a moratorium, protected forest, or conservation area under national/regional law, and whether there are reforesta

# Sanity Check - Retrieve All Nodes from VectorIndex in Chroma Collection

In [ ]:
def retrieve_nodes_by_filename(collection_name, file_name, chroma_client):
    """
    Retrieve all nodes from a ChromaDB collection where the metadata "file_name" matches a specific value.
    
    Args:
        collection_name (str): Name of the ChromaDB collection
        file_name (str): File name to match in metadata
        chroma_client: ChromaDB client
        
    Returns:
        list: List of nodes matching the file name
    """
    try:
        # Get the collection
        collection = chroma_client.get_collection(name=collection_name)
        
        # Query the collection for documents with the specified file_name in metadata
        results = collection.get(
            where={"file_name": file_name},
            include=["metadatas", "documents"]
        )
        
        print(f"Found {len(results['ids'])} nodes matching file_name: {file_name}")
        
        # Format the results for better readability
        formatted_results = []
        for i in range(len(results['ids'])):
            node_data = {
                "id": results['ids'][i],
                "metadata": results['metadatas'][i] if i < len(results['metadatas']) else {},
                "document": results['documents'][i] if i < len(results['documents']) else ""
            }
            formatted_results.append(node_data)
        
        return formatted_results
        
    except Exception as e:
        print(f"Error retrieving nodes: {e}")
        return []

# # Use the function to retrieve nodes for the specified file
# file_name = "VCS_3226_302541_Padang Tikar_REDD_PDD_VCS_v1.0_clean (1).pdf"
# collection_name = "pdd_new"

# nodes = retrieve_nodes_by_filename(collection_name, file_name, chroma_client)

# # Print the full text of each node
# if nodes:
#     print(f"\nFull text content of all {len(nodes)} nodes:\n")
#     for i, node in enumerate(nodes):
#         print(f"--- Node {i+1} ---")
#         print(f"ID: {node['id']}")
#         print(f"Metadata: {json.dumps(node['metadata'], indent=2)}")
#         print(f"Document content:\n{node['document']}")
#         print("-" * 80)
# else:
#     print("No nodes found matching the criteria.")

# Upload to Database

In [ ]:
from upload_to_supabase import (
    upload_project_info, 
    upload_project_summary, 
    upload_risk_metrics,
    upload_time_series_data,
    upload_pie_chart_data,
    upload_geo_data
)